# Interactive Recruitment Dashboard for Data Science Practitioners

This notebook demonstrates how to analyze recruitment data for Data Science positions using our `recruitment_functions` module. The notebook shows:

1. How to use the centralized functions for data loading and analysis
2. Interactive visualizations of recruitment metrics
3. Sample data generation and management
4. Dashboard integration

The actual dashboard implementation is in `dashboard.py`, which uses the same functions we'll explore here.

## 1. Setup and Imports

First, let's import our centralized functions and required libraries:

In [ ]:
# Import our centralized functions
from recruitment_functions import (
    create_sample_data,
    load_recruitment_data,
    create_application_funnel,
    create_source_effectiveness,
    analyze_experience_education
)

# Additional libraries for visualization
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Optional: Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 2. Create Sample Data

Let's create some sample recruitment data in our SQLite database. This is handled by the `create_sample_data()` function:

In [ ]:
# Create sample data (no-op if data already exists)
create_sample_data()

# Load the data to verify
df = load_recruitment_data()
print(f"Loaded {len(df)} recruitment records")

## 3. Analyze Recruitment Funnel

Let's visualize the application funnel to see conversion rates at each stage:

In [ ]:
# Create and display the funnel visualization
funnel_fig = create_application_funnel(df)
funnel_fig.show()

## 4. Source Effectiveness Analysis

Let's analyze which recruitment sources are most effective:

In [ ]:
# Create the source effectiveness visualization
source_fig, source_metrics = create_source_effectiveness(df)

# Display the chart
source_fig.show()

# Display the metrics table
print("\nDetailed Source Metrics:")
print(source_metrics.to_string(index=False))

## 5. Candidate Qualifications Analysis

Finally, let's analyze the relationship between candidate experience, education, and success rates:

In [ ]:
# Get qualification analysis
qual_analysis = analyze_experience_education(df)

# Display with better formatting
pd.set_option('display.float_format', '{:.2f}'.format)
display(qual_analysis.style.background_gradient(subset=['hire_rate'], cmap='YlGn'))

## 1. Environment Setup & Dependencies

First, we'll install and import all the required libraries for our dashboard. We'll use:
- `pandas` and `numpy` for data manipulation
- `sqlalchemy` for database connections
- `plotly` for interactive visualizations
- `streamlit` for creating the dashboard
- `pytest` for testing
- `python-dotenv` for environment management

In [ ]:
# Install required packages
!pip install pandas numpy sqlalchemy plotly streamlit pytest python-dotenv streamlit-echarts

# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import streamlit as st
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 2. Database Connection Setup

We'll set up a connection to our database using SQLAlchemy. For this example, we'll create a sample SQLite database with recruitment data. In a production environment, you would typically connect to a PostgreSQL or MySQL database.

In [ ]:
# Create a SQLite database in the current workspace
import os

db_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'recruitment.db')
engine = create_engine(f'sqlite:///{db_path}')

## 3. Generate Sample Data

Let's create some sample recruitment data to populate our database. We'll generate realistic data for candidates, interviews, job postings, and applications.

In [ ]:
# Generate sample data with error handling
import random
from datetime import datetime, timedelta

try:
    # Helper functions for generating random dates
    def random_date(start_date, end_date):
        time_between = end_date - start_date
        days_between = time_between.days
        random_days = random.randrange(days_between)
        return start_date + timedelta(days=random_days)

    # Sample data parameters
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2025, 11, 1)
    sources = ['LinkedIn', 'Company Website', 'Referral', 'Job Board', 'University Partnership']
    education_levels = ['Bachelor's', 'Master's', 'PhD', 'Bootcamp']
    interview_stages = ['Technical Screen', 'Coding Challenge', 'System Design', 'Cultural Fit', 'Final Round']
    locations = ['Remote', 'New York', 'San Francisco', 'London', 'Singapore']

    # Generate candidates data
    candidates_data = []
    for i in range(100):
        candidates_data.append({
            'candidate_id': i + 1,
            'name': f'Candidate {i+1}',
            'email': f'candidate{i+1}@example.com',
            'source': random.choice(sources),
            'application_date': random_date(start_date, end_date),
            'years_experience': random.randint(0, 15),
            'education_level': random.choice(education_levels)
        })

    candidates_df = pd.DataFrame(candidates_data)

    # Generate job postings
    job_titles = ['Data Scientist', 'Senior Data Scientist', 'ML Engineer', 'Data Science Manager']
    departments = ['Analytics', 'Research', 'Product', 'Engineering']

    job_postings_data = []
    for i in range(10):
        job_postings_data.append({
            'job_id': i + 1,
            'title': random.choice(job_titles),
            'department': random.choice(departments),
            'location': random.choice(locations),
            'required_skills': 'Python, SQL, Machine Learning',
            'posting_date': random_date(start_date, end_date),
            'status': random.choice(['Open', 'Closed', 'On Hold'])
        })

    job_postings_df = pd.DataFrame(job_postings_data)

    # Generate applications with better status distribution
    application_statuses = {
        'Applied': 0.3,
        'Screening': 0.2,
        'Interviewing': 0.2,
        'Offered': 0.1,
        'Hired': 0.1,
        'Rejected': 0.1
    }

    applications_data = []
    for candidate in candidates_data:
        num_applications = random.randint(1, 3)
        for _ in range(num_applications):
            status = random.choices(
                list(application_statuses.keys()),
                weights=list(application_statuses.values())
            )[0]
            applications_data.append({
                'application_id': len(applications_data) + 1,
                'candidate_id': candidate['candidate_id'],
                'job_id': random.randint(1, 10),
                'apply_date': candidate['application_date'],
                'status': status
            })

    applications_df = pd.DataFrame(applications_data)

    # Save data to database with error handling
    try:
        candidates_df.to_sql('candidates', engine, if_exists='replace', index=False)
        job_postings_df.to_sql('job_postings', engine, if_exists='replace', index=False)
        applications_df.to_sql('applications', engine, if_exists='replace', index=False)
        print("✅ Sample data generated and loaded into the database successfully!")
    except Exception as e:
        print(f"❌ Error saving to database: {str(e)}")

except Exception as e:
    print(f"❌ Error generating sample data: {str(e)}")

## 4. Create Dashboard Functions

Now let's create functions to calculate key recruitment metrics and create interactive visualizations. We'll focus on:
1. Application funnel analysis
2. Time-to-hire metrics
3. Source effectiveness
4. Candidate qualification analysis

In [ ]:
# Define helper functions for the dashboard
@st.cache_data(ttl=600)  # Cache data for 10 minutes
def load_recruitment_data():
    """Load and join relevant recruitment data"""
    try:
        query = """
        SELECT 
            a.application_id,
            a.status as application_status,
            a.apply_date,
            c.candidate_id,
            c.source,
            c.years_experience,
            c.education_level,
            j.title as job_title,
            j.department,
            j.location
        FROM applications a
        JOIN candidates c ON a.candidate_id = c.candidate_id
        JOIN job_postings j ON a.job_id = j.job_id
        """
        return pd.read_sql(query, engine)
    except Exception as e:
        st.error(f"Error loading data: {str(e)}")
        return pd.DataFrame()

@st.cache_data(ttl=600)
def create_application_funnel():
    """Create application funnel visualization"""
    try:
        data = load_recruitment_data()
        if data.empty:
            return None
            
        funnel_data = data['application_status'].value_counts().reset_index()
        funnel_data.columns = ['Stage', 'Count']
        
        # Sort stages in logical order
        stage_order = ['Applied', 'Screening', 'Interviewing', 'Offered', 'Hired', 'Rejected']
        funnel_data['Stage'] = pd.Categorical(funnel_data['Stage'], categories=stage_order, ordered=True)
        funnel_data = funnel_data.sort_values('Stage')
        
        fig = go.Figure(go.Funnel(
            y=funnel_data['Stage'],
            x=funnel_data['Count'],
            textinfo="value+percent initial",
            textposition="inside",
            textfont=dict(size=14)
        ))
        
        fig.update_layout(
            title="Application Funnel",
            showlegend=False,
            height=400
        )
        return fig
    except Exception as e:
        st.error(f"Error creating funnel chart: {str(e)}")
        return None

@st.cache_data(ttl=600)
def create_source_effectiveness(department_filter=None):
    """Analyze effectiveness of different recruitment sources"""
    try:
        data = load_recruitment_data()
        if data.empty:
            return None
            
        if department_filter and department_filter != "All":
            data = data[data['department'] == department_filter]
            
        source_metrics = data.groupby('source').agg({
            'application_id': 'count',
            'application_status': lambda x: (x == 'Hired').sum()
        }).reset_index()
        
        source_metrics.columns = ['Source', 'Total Applications', 'Hires']
        source_metrics['Conversion Rate'] = (source_metrics['Hires'] / source_metrics['Total Applications'] * 100).round(2)
        source_metrics = source_metrics.sort_values('Conversion Rate', ascending=False)
        
        fig = px.bar(source_metrics, 
                     x='Source', 
                     y=['Total Applications', 'Hires'],
                     barmode='group',
                     title='Recruitment Source Effectiveness',
                     labels={'value': 'Count', 'variable': 'Metric'},
                     color_discrete_sequence=['#636EFA', '#00CC96'])
                     
        fig.update_layout(
            height=400,
            xaxis_title="Source",
            yaxis_title="Count",
            hovermode='x unified'
        )
        return fig, source_metrics
    except Exception as e:
        st.error(f"Error analyzing source effectiveness: {str(e)}")
        return None, None

@st.cache_data(ttl=600)
def analyze_experience_education():
    """Analyze success rates by experience and education"""
    try:
        data = load_recruitment_data()
        if data.empty:
            return None
            
        # Create experience bins
        data['experience_range'] = pd.cut(
            data['years_experience'],
            bins=[0, 2, 5, 8, float('inf')],
            labels=['0-2 years', '3-5 years', '6-8 years', '8+ years']
        )
        
        exp_edu_metrics = pd.pivot_table(
            data,
            index=['experience_range', 'education_level'],
            values='application_id',
            columns='application_status',
            aggfunc='count',
            fill_value=0
        ).reset_index()
        
        # Calculate hire rate
        exp_edu_metrics['Hire Rate'] = (
            exp_edu_metrics['Hired'] / 
            exp_edu_metrics[['Applied', 'Screening', 'Interviewing', 'Offered', 'Hired', 'Rejected']].sum(axis=1) * 100
        ).round(2)
        
        return exp_edu_metrics
    except Exception as e:
        st.error(f"Error analyzing experience and education: {str(e)}")
        return None

## 5. Create Streamlit Dashboard

Now let's create the main Streamlit dashboard application. We'll create an interactive interface with filters and visualizations.

In [ ]:
# Save this code as 'dashboard.py'
def create_dashboard():
    """Create the main dashboard application"""
    
    dashboard_code = """
import streamlit as st
import plotly.graph_objects as go
from datetime import datetime, timedelta
import pandas as pd
from sqlalchemy import create_engine
import os
from recruitment_functions import *

# Set page config
st.set_page_config(
    page_title="Data Science Recruitment Dashboard",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for better styling
st.markdown('''
    <style>
    .metric-card {
        background-color: #f0f2f6;
        border-radius: 10px;
        padding: 20px;
        text-align: center;
    }
    .big-number {
        font-size: 24px;
        font-weight: bold;
        color: #0068c9;
    }
    </style>
''', unsafe_allow_html=True)

# Title and description
st.title("📊 Data Science Recruitment Dashboard")
st.markdown("""
This dashboard provides real-time insights into the Data Science recruitment pipeline.
Track application funnel, source effectiveness, and candidate qualifications.
""")

# Initialize session state for filters
if 'date_filter' not in st.session_state:
    st.session_state.date_filter = (datetime.now() - timedelta(days=180), datetime.now())
if 'department_filter' not in st.session_state:
    st.session_state.department_filter = "All"

# Sidebar filters
with st.sidebar:
    st.header("📌 Filters")
    
    # Date range filter
    st.session_state.date_filter = st.date_input(
        "Date Range",
        value=st.session_state.date_filter
    )
    
    # Department filter
    data = load_recruitment_data()
    if not data.empty:
        departments = ["All"] + sorted(data['department'].unique().tolist())
        st.session_state.department_filter = st.selectbox(
            "Department",
            departments
        )
    
    # Add filter information
    st.sidebar.info("Filters are applied to all visualizations automatically")

# Load data with selected filters
try:
    data = load_recruitment_data()
    if not data.empty:
        # Apply filters
        if st.session_state.department_filter != "All":
            data = data[data['department'] == st.session_state.department_filter]
        
        # Main dashboard layout
        col1, col2 = st.columns(2)

        with col1:
            # Application Funnel
            funnel = create_application_funnel()
            if funnel:
                st.plotly_chart(funnel, use_container_width=True)
            
        with col2:
            # Source Effectiveness
            source_chart, source_metrics = create_source_effectiveness(st.session_state.department_filter)
            if source_chart:
                st.plotly_chart(source_chart, use_container_width=True)

        # Key Metrics
        st.header("📈 Key Recruitment Metrics")
        metrics_cols = st.columns(4)
        
        with metrics_cols[0]:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric(
                "Total Applications",
                len(data),
                delta=f"{len(data)-100} vs prev. period"
            )
            st.markdown('</div>', unsafe_allow_html=True)

        with metrics_cols[1]:
            conversion = (len(data[data['application_status'] == 'Hired']) / len(data) * 100).round(2)
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric(
                "Conversion Rate",
                f"{conversion}%",
                delta=f"{(conversion-10):.1f}% vs target"
            )
            st.markdown('</div>', unsafe_allow_html=True)

        with metrics_cols[2]:
            avg_exp = data['years_experience'].mean().round(1)
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric(
                "Avg. Years of Experience",
                avg_exp,
                delta=f"{(avg_exp-5):.1f} vs target"
            )
            st.markdown('</div>', unsafe_allow_html=True)

        with metrics_cols[3]:
            top_source = data['source'].mode()[0]
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric(
                "Top Recruitment Source",
                top_source
            )
            st.markdown('</div>', unsafe_allow_html=True)

        # Candidate Qualifications Analysis
        st.header("👥 Candidate Qualifications Analysis")
        qual_analysis = analyze_experience_education()
        if qual_analysis is not None:
            st.dataframe(
                qual_analysis.style.background_gradient(subset=['Hire Rate'], cmap='YlGn'),
                use_container_width=True
            )
            
        # Download section
        st.header("📥 Export Data")
        if st.button("Download Recruitment Report"):
            csv = data.to_csv(index=False)
            st.download_button(
                label="Download CSV",
                data=csv,
                file_name="recruitment_data.csv",
                mime="text/csv"
            )
    else:
        st.error("No data available. Please check the database connection.")
        
except Exception as e:
    st.error(f"An error occurred: {str(e)}")
    st.info("Please make sure the database is properly set up and contains recruitment data.")
"""

    # Save the dashboard code
    with open('dashboard.py', 'w') as f:
        f.write(dashboard_code)
    
    print("✅ Dashboard code has been saved to 'dashboard.py'")
    print("📊 To run the dashboard, use the command: streamlit run dashboard.py")

# Create the dashboard file
create_dashboard()

## Running the Dashboard

To run the dashboard:

1. Make sure all requirements are installed:
```bash
pip install streamlit pandas plotly sqlalchemy
```

2. Save the dashboard code to a file named `dashboard.py`

3. Run the dashboard using:
```bash
streamlit run dashboard.py
```

The dashboard will open in your default web browser, showing:
- Interactive application funnel
- Source effectiveness analysis
- Candidate qualification metrics
- Key recruitment KPIs

You can use the sidebar filters to:
- Select date ranges
- Filter by department
- Focus on specific job titles
- Analyze different candidate sources

The dashboard automatically updates when you change the filters, providing real-time insights into your recruitment pipeline.